# Modify Forcing data

This notebook provides tools to load, modify and save forcing input (Simstrat-like `Forcing.dat`)

## Content:

- Load forcing data
- Modify forcing data
- Save modified forcing data

## Available functions:

- `data, units = load_forcing_data(path, file, start='file_start', stop='file_stop', load_units=True)` — Load forcing data and optionally load units of forcing variables
- `save_forcing_data(data, local_path, file_name, copy=True)` — Save modified forcing data to a new or a existing file
- `data = check_negative(data, val_save, var)` — Check for negative values for a forcing variable

---

## Import libraries

In [1]:
# Import standard Python libraries
import os
import numpy as np
import pandas as pd

# Import custom functions
from functions import load_forcing_data, save_forcing_data, check_negative

## Load forcing data

You can change these parameters:
- `local_path`: the path to the forcing data file
- `file_name`: the name of the forcing data file
    - usually `Forcing.dat`
- `pos_vars`: variables that cannot be negative
- `start`: from when on to edit forcing data
  - date and time in format `'YYYYMMDDHHMM'` (Year-Month-Day-Hour-Minute)
  - example: `'202510231500'` = October 23, 2025 at 15:00 (3:00 PM)
  - default: `'file_start'` -> start at beginning of file
- `stop`: until when to edit forcing data
  - same format as `start`
  - default: `'file_stop'` -> stop at end of file
- `load_units`: whether to load the units from the file
    - `True` to load the units (default)
    - `False` to not load the units, returns an empty DataFrame to `units`

In [2]:
# Set parameters
local_path = '../../tests/TestCase_LakeZurich'
file_name = 'MeteoForcing.dat'
pos_vars = ['sol', 'vap', 'cloud', 'rain']

# Load data
data, units = load_forcing_data(
    path = local_path,
    file = file_name,
    start = 'file_start',
    stop = 'file_stop',
    load_units = True
)

# Display units and data
display(units)
display(data.round(4))

Loading data from local file: ../../tests/TestCase_LakeZurich\MeteoForcing.dat
✓ Loaded 367704 rows from ../../tests/TestCase_LakeZurich\MeteoForcing.dat
✓ Filtered to date range: 367704 rows between 1981-01-01 and 2022-12-12


,u,v,Tair,sol,vap,cloud,rain
Unit,m/s,m/s,°C,W/m2,mbar,-,m/hr


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,0.1006,1.2783,-0.2750,0.0,5.5395,0.0,0.0
1,1981-01-01 01:00:02,-0.9830,0.0897,-1.1679,0.0,5.3331,0.0,0.0
2,1981-01-01 01:59:57,0.1023,0.7836,-0.7711,0.0,5.1267,0.0,0.0
3,1981-01-01 03:00:00,3.2412,2.5695,3.8922,0.0,4.3011,0.0,0.0
4,1981-01-01 04:00:02,3.6595,1.4582,4.3883,1.0,3.9914,0.0,0.0
...,...,...,...,...,...,...,...,...
367699,2022-12-12 19:00:02,-1.5380,-0.4410,-5.8970,1.0,3.2000,0.0,0.0
367700,2022-12-12 19:59:57,-1.6000,-0.0000,-6.3970,1.0,3.2000,0.0,0.0
367701,2022-12-12 21:00:00,-1.1535,-0.3308,-6.1970,1.0,3.3000,0.0,0.0
367702,2022-12-12 22:00:02,-1.0718,-0.2474,-6.2970,1.0,3.4000,0.0,0.0


## Modify forcing data

You can modify the data in the following ways:
- `Add a constant`
- `Multiply by a constant`
- `Set to a constant`
- `Add a linear trend`

Be aware that:
- The variables in pos_vars cannot be negative
    - this is controlled in this code

### Add constant

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `constant`: the constant to add
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative

In [3]:
# Set parameters
variable = 'u'
constant = -1

# Modify and display data
val_save = data[variable].copy()
data[variable] += float(constant)
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'u' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,-0.8994,1.2783,-0.2750,0.0,5.5395,0.0,0.0
1,1981-01-01 01:00:02,-1.9830,0.0897,-1.1679,0.0,5.3331,0.0,0.0
2,1981-01-01 01:59:57,-0.8977,0.7836,-0.7711,0.0,5.1267,0.0,0.0
3,1981-01-01 03:00:00,2.2412,2.5695,3.8922,0.0,4.3011,0.0,0.0
4,1981-01-01 04:00:02,2.6595,1.4582,4.3883,1.0,3.9914,0.0,0.0
...,...,...,...,...,...,...,...,...
367699,2022-12-12 19:00:02,-2.5380,-0.4410,-5.8970,1.0,3.2000,0.0,0.0
367700,2022-12-12 19:59:57,-2.6000,-0.0000,-6.3970,1.0,3.2000,0.0,0.0
367701,2022-12-12 21:00:00,-2.1535,-0.3308,-6.1970,1.0,3.3000,0.0,0.0
367702,2022-12-12 22:00:02,-2.0718,-0.2474,-6.2970,1.0,3.4000,0.0,0.0


### Multiply by constant

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `constant`: the constant to multiply by
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative

In [4]:
# Set parameters
variable = 'v'
constant = 1/10

# Modify and display data
val_save = data[variable].copy()
data[variable] *= float(constant)
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'v' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,-0.8994,0.1278,-0.2750,0.0,5.5395,0.0,0.0
1,1981-01-01 01:00:02,-1.9830,0.0090,-1.1679,0.0,5.3331,0.0,0.0
2,1981-01-01 01:59:57,-0.8977,0.0784,-0.7711,0.0,5.1267,0.0,0.0
3,1981-01-01 03:00:00,2.2412,0.2570,3.8922,0.0,4.3011,0.0,0.0
4,1981-01-01 04:00:02,2.6595,0.1458,4.3883,1.0,3.9914,0.0,0.0
...,...,...,...,...,...,...,...,...
367699,2022-12-12 19:00:02,-2.5380,-0.0441,-5.8970,1.0,3.2000,0.0,0.0
367700,2022-12-12 19:59:57,-2.6000,-0.0000,-6.3970,1.0,3.2000,0.0,0.0
367701,2022-12-12 21:00:00,-2.1535,-0.0331,-6.1970,1.0,3.3000,0.0,0.0
367702,2022-12-12 22:00:02,-2.0718,-0.0247,-6.2970,1.0,3.4000,0.0,0.0


### Set to constant

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `constant`: the constant value of the variable
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative

In [5]:
# Set parameters
variable = 'Tair'
constant = 10

# Modify and display data
val_save = data[variable].copy()
data[variable] = np.full((len(data)), float(constant))
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'Tair' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,-0.8994,0.1278,10.0,0.0,5.5395,0.0,0.0
1,1981-01-01 01:00:02,-1.9830,0.0090,10.0,0.0,5.3331,0.0,0.0
2,1981-01-01 01:59:57,-0.8977,0.0784,10.0,0.0,5.1267,0.0,0.0
3,1981-01-01 03:00:00,2.2412,0.2570,10.0,0.0,4.3011,0.0,0.0
4,1981-01-01 04:00:02,2.6595,0.1458,10.0,1.0,3.9914,0.0,0.0
...,...,...,...,...,...,...,...,...
367699,2022-12-12 19:00:02,-2.5380,-0.0441,10.0,1.0,3.2000,0.0,0.0
367700,2022-12-12 19:59:57,-2.6000,-0.0000,10.0,1.0,3.2000,0.0,0.0
367701,2022-12-12 21:00:00,-2.1535,-0.0331,10.0,1.0,3.3000,0.0,0.0
367702,2022-12-12 22:00:02,-2.0718,-0.0247,10.0,1.0,3.4000,0.0,0.0


### Add linear trend

You can change these parameters:
- `variable`: choose the variable to modify from:
    - `'u'`, `'v'`, `'Tair'`, `'sol'`, `'vap'`, `'cloud'`, `'rain'`
- `step`: the time of change
    - any positive integer
    - usually `1` corresponds to one hour
    - accordingly `24` corresponds to one day
    -  `8760` corresponds to one year
- `slope`: the amount of change per time of change
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative
    - `variable` increases by `slope` every `step` hours
- `offset`: the offset to the trend
    - any real number (can also be negative and/or a fraction)
    - variables in pos_vars cannot become negative
    - default is `0`

In [6]:
# Set parameters
variable = 'Tair'
step = 8760
slope = 1/43
offset = 0

# Modify and display data
rate = float(slope / step)
val_save = data[variable].copy()
data[variable] = (data[variable] +
    np.linspace(0, rate * (len(data) - 1), len(data)) +
    float(offset))
if variable in pos_vars:
    data[variable] = check_negative(
    data[variable], val_save, variable)
else:
    print(f"✓ Forcing variable '{variable}' modified")
display(data.round(4))

✓ Forcing variable 'Tair' modified


,Datetime,u,v,Tair,sol,vap,cloud,rain
0,1981-01-01 00:00:00,-0.8994,0.1278,10.0000,0.0,5.5395,0.0,0.0
1,1981-01-01 01:00:02,-1.9830,0.0090,10.0000,0.0,5.3331,0.0,0.0
2,1981-01-01 01:59:57,-0.8977,0.0784,10.0000,0.0,5.1267,0.0,0.0
3,1981-01-01 03:00:00,2.2412,0.2570,10.0000,0.0,4.3011,0.0,0.0
4,1981-01-01 04:00:02,2.6595,0.1458,10.0000,1.0,3.9914,0.0,0.0
...,...,...,...,...,...,...,...,...
367699,2022-12-12 19:00:02,-2.5380,-0.0441,10.9762,1.0,3.2000,0.0,0.0
367700,2022-12-12 19:59:57,-2.6000,-0.0000,10.9762,1.0,3.2000,0.0,0.0
367701,2022-12-12 21:00:00,-2.1535,-0.0331,10.9762,1.0,3.3000,0.0,0.0
367702,2022-12-12 22:00:02,-2.0718,-0.0247,10.9762,1.0,3.4000,0.0,0.0


# Step 4
## Save modified forcing data

You can change this parameter:
- `copy`: Wheter to create a new forcing file (with _modified appended to the file_name) or to overwrite the original
    - `True` to create a new file (default)
    - `False` to overwrite the original

In [7]:
# Save data
save_forcing_data(
    data,
    path = local_path,
    file = file_name,
    copy = True
)

Writing modified data into: ../../tests/TestCase_LakeZurich\MeteoForcing_modified.dat
✓ Modified 367704 rows
